# ASG Airlines — Flight Duration & Overnight Handling

**Stage:** Silver (updates `silver/flights` in place)

## Why this notebook exists, and why it uses full-datetime math

An earlier version of this logic computed duration by comparing only the **time-of-day** component of `departure_time`/`arrival_time`, ignoring the date entirely, and applying a `+1440` minute correction whenever the arrival clock-time was earlier than the departure clock-time. That approach found 122 "overnight" flights — but it was actually **masking a real data error**: one record (`SJ192`) had an arrival *date* one day before its departure date, which is impossible. The time-of-day-only method silently treated it as a normal ~5 hour flight instead of flagging it.

Before trusting either method, we verified whether the date component in this dataset is meaningful or just placeholder noise — it turned out to be meaningful: in 122 of 123 cases where departure and arrival dates differ, the arrival date is legitimately `departure_date + 1 day` (a real overnight flight). Only `SJ192` was the odd one out, with an arrival date *before* departure.

**Conclusion: full-datetime subtraction is the correct approach.** It naturally handles genuine overnight flights (since both fields carry a real date) without needing any manual day-rollover correction, and it correctly exposes `SJ192` as a **negative-duration anomaly** rather than smoothing it into a plausible-looking flight.

## Logic implemented

1. `duration_minutes` = `(arrival_time − departure_time)` in total minutes, using the full datetime (date included).
2. `is_overnight_flight` = `True` when `duration_minutes > 0` **and** the departure/arrival dates differ (a genuine overnight flight).
3. `duration_is_anomalous` = `True` when `duration_minutes <= 0` (impossible — arrival at or before departure) **or** `> 1440` (over 24 hours, unrealistic for this dataset). Anomalous rows are **flagged, not dropped or corrected** — they stay visible for inspection on the dashboard's anomaly page.
4. `duration_formatted` — a human-readable string (`"2h 15m"`, or `"-1140m (INVALID)"` for the anomaly) for dashboard readability.

## Step 1 — Storage configuration

In [1]:
import os
import pandas as pd
import numpy as np
from deltalake import write_deltalake, DeltaTable

STORAGE_ACCOUNT_NAME = "stasgairlines01"
CONTAINER_SILVER     = "silver"
TABLE_FLIGHTS        = "flights"

STORAGE_KEY = os.environ.get("ADLS_STORAGE_KEY", "")
if not STORAGE_KEY:
    try:
        import subprocess
        res = subprocess.run(
            ["az", "storage", "account", "keys", "list",
             "--account-name", STORAGE_ACCOUNT_NAME,
             "--resource-group", "rg-asg-airlines",
             "--query", "[0].value", "-o", "tsv"],
            capture_output=True, text=True, check=True
        )
        STORAGE_KEY = res.stdout.strip()
    except Exception:
        pass

if not STORAGE_KEY:
    raise RuntimeError("ADLS_STORAGE_KEY environment variable is not set.")

storage_options = {
    "azure_storage_account_name": STORAGE_ACCOUNT_NAME,
    "azure_storage_access_key": STORAGE_KEY,
}

silver_uri = f"az://{CONTAINER_SILVER}/{TABLE_FLIGHTS}"
print(f"Target Delta table: {silver_uri}")

Target Delta table: az://silver/flights


## Step 2 — Read `silver/flights`

Also drops any transient columns left over from earlier iterations of this logic (e.g. a discarded time-of-day-only calculation), so re-running this notebook stays idempotent.

In [1]:
dt_flights = DeltaTable(silver_uri, storage_options=storage_options)
df_flights = dt_flights.to_pandas()

unwanted_cols = ["__index_level_0__", "raw_diff_minutes", "tod_diff_minutes"]
for col in unwanted_cols:
    if col in df_flights.columns:
        df_flights = df_flights.drop(columns=[col])

print(f"Read {len(df_flights):,} rows from {silver_uri}")

Read 1,003 rows from az://silver/flights


## Step 3 — Recompute duration using full datetime timestamps

In [1]:
dep_dt = pd.to_datetime(df_flights["departure_time"], format="mixed")
arr_dt = pd.to_datetime(df_flights["arrival_time"], format="mixed")

# Full datetime difference in minutes (date included, no manual rollover needed)
duration_minutes = (arr_dt - dep_dt).dt.total_seconds() / 60.0
duration_minutes = duration_minutes.round(2)

# Genuine overnight: positive duration AND the calendar date actually changed
dep_date = dep_dt.dt.date
arr_date = arr_dt.dt.date
is_overnight = (duration_minutes > 0) & (dep_date != arr_date)

# Anomaly: zero/negative duration, or an unrealistically long "flight"
duration_is_anomalous = (duration_minutes <= 0) | (duration_minutes > 1440)

def format_duration(mins):
    if mins <= 0:
        return f"{mins:.0f}m (INVALID)"
    total_mins = int(round(mins))
    h = total_mins // 60
    m = total_mins % 60
    return f"{h}h {m}m"

duration_formatted = duration_minutes.apply(format_duration)

df_flights["is_overnight_flight"] = is_overnight
df_flights["duration_minutes"] = duration_minutes
df_flights["duration_is_anomalous"] = duration_is_anomalous
df_flights["duration_formatted"] = duration_formatted

print("Duration, overnight, and anomaly columns computed.")

Duration, overnight, and anomaly columns computed.


## Step 4 — Validate against the known `SJ192` anomaly, and report summary statistics

In [1]:
print("=" * 80)
print("FLIGHT DURATION VALIDATION REPORT")
print("=" * 80)

sj192_row = df_flights[df_flights["flight_id"] == "SJ192"]
if len(sj192_row) > 0:
    r = sj192_row.iloc[0]
    print("\nTarget validation case (SJ192):")
    print(f"  departure_time        : '{r['departure_time']}'")
    print(f"  arrival_time          : '{r['arrival_time']}'")
    print(f"  duration_minutes      : {r['duration_minutes']} min")
    print(f"  is_overnight_flight   : {r['is_overnight_flight']}")
    print(f"  duration_is_anomalous : {r['duration_is_anomalous']}  (expected True)")
    print(f"  duration_formatted    : '{r['duration_formatted']}'")

total_overnight_cnt = int(is_overnight.sum())
total_anomalous_cnt = int(duration_is_anomalous.sum())

print(f"\nTotal rows: {len(df_flights):,}")
print(f"  Overnight flights : {total_overnight_cnt:,} ({total_overnight_cnt / len(df_flights) * 100:.2f}%)")
print(f"  Anomalous flights : {total_anomalous_cnt:,} ({total_anomalous_cnt / len(df_flights) * 100:.2f}%)")

stats = duration_minutes.describe()
print("\nDuration distribution (minutes):")
print(f"  Min={stats['min']:.1f}  25%={stats['25%']:.1f}  Median={stats['50%']:.1f}  Mean={stats['mean']:.1f}  75%={stats['75%']:.1f}  Max={stats['max']:.1f}")

FLIGHT DURATION VALIDATION REPORT

Target validation case (SJ192):
  departure_time        : '2026-04-19 18:45:42'
  arrival_time          : '2026-04-18 23:45:42'
  duration_minutes      : -1140.0 min
  is_overnight_flight   : False
  duration_is_anomalous : True  (expected True)
  duration_formatted    : '-1140m (INVALID)'

Total rows: 1,003
  Overnight flights : 122 (12.16%)
  Anomalous flights : 1 (0.10%)

Duration distribution (minutes):
  Min=-1140.0  25%=99.0  Median=166.0  Mean=163.2  75%=232.5  Max=300.0


## Step 5 — Overwrite `silver/flights` with the corrected columns

In [1]:
write_deltalake(silver_uri, df_flights, mode="overwrite", schema_mode="overwrite", storage_options=storage_options)
print(f"Wrote {len(df_flights):,} rows -> {silver_uri}")

Wrote 1,003 rows -> az://silver/flights


## Step 6 — Verify

In [1]:
dt_verify = DeltaTable(silver_uri, storage_options=storage_options)
df_verify = dt_verify.to_pandas()
print(f"[Verified] {silver_uri}: {len(df_verify):,} rows | Schema fields: {len(dt_verify.schema().fields)}")
print(f"Final schema: {[f.name for f in dt_verify.schema().fields]}")

print("\nDuration & overnight recomputation complete.")

[Verified] az://silver/flights: 1,003 rows | Schema fields: 13
Final schema: ['flight_id', 'airline', 'source', 'destination', 'departure_time', 'arrival_time', 'ingestion_timestamp', 'source_file', 'airline_is_missing', 'is_overnight_flight', 'duration_minutes', 'duration_is_anomalous', 'duration_formatted']

Duration & overnight recomputation complete.
